# CRUD, formulaires, gestions d'utilisateurs


In [1]:
# on restaure notre base de données en remplaçant la BDD par son backup 
!cp ./richelieu.db.bak ./richelieu.db

On sait maintenant faire **toutes les opérations CRUD via une application Flask**. On peut donc utiliser ensemble Flask, SQLAlchemy, WTForms et les templates Jinja pour faire presque tout ce qui est attendu d'une appli Web.

Pour ce dernier cours, on va voir:
- comment **créer et gérer des comptes utilisateurs**
- comment **tester une appli Flask**
- comment **écrire une API**


---

# Les `users`

![db schema](./img/db_schema.png)

Votre regard aguisé aura remarqué que pour le moment, on a pas modélisé la table `user`. Elle sert à:
- ajouter de la gestion d'utilisateur.ice.s sur le site (créer un compte utilisateur et se connecter)
- limiter l'accès à certaines pages aux utilisateur.ice.s connecté.e.s. Par exemple, **on ne pourra modifier la base de données que si on est connecté.e**.

On remarque aussi que `user` n'est liée à aucune autre table: elle sert seulement à stocker les utilisateur.ice.s.

## Le modèle de `User`

Un compte utilisateur, c'est simplement une ligne de la table `user`.

**Voici notre table `User`** de base. 

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
```

Normalement, il existe déjà un compte utilisateur qui a pour mail `admin@mail.com` et pour mot de passe `admin`.


In [7]:
# on recrée notre appli, et notre modèle `User`
from typing import List, Optional
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash
from sqlalchemy import ForeignKey
from sqlalchemy.orm import Mapped, mapped_column

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)

class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]

# on fait une requête pour voir les utilisateur.ice.s:
with app.app_context():
    users = db.session.execute(db.select(User)).scalars().all()

    print("Nombre d'utilisateur.ice.s:", len(users))

    user = users[0]
    print(f"nom: {user.user_name}, mail:{user.user_mail}, mdp: {user.user_password}")

/home/paul/Documents/cours/tnah_devapp/richelieu.db
Nombre d'utilisateur.ice.s: 1
nom: admin, mail:admin@mail.com, mdp: scrypt:32768:8:1$ttz8dh2PA9LayiiT$fee7c74bf665c4ba8fd2937297a0c968308d39db17394742ec52a7fd67ff05cd3cb62314a4c65fba0e74a6bd89d1cec9be248213f6fb06c71d1cf0dab7db2651


## Sécurité: stocker les mots de passe

Le mot de passe affiché au dessus (`scrypt:...`), c'est le mot de passe tel qu'il est enregistré dans la base de données. Cette énorme chaîne de caractères **n'est pas le mot de passe tel que définit pour l'utilisateur, mais un `hash` du mot de passe** (le "vrai" mot de passe utilisé pour se connecter, c'est `admin`). 

### Hashage ?

Pourquoi ce qui est stocké en base n'est pas le vrai mot de passe ? Parce que **⚠️⚠️⚠️ ON NE STOCKE JAMAIS UN MOT DE PASSE EN BRUT DANS UNE BASE DE DONNÉES ⚠️⚠️⚠️**. Quand on créée un compte en ligne, **c'est un hash du mot de passe qui est stocké**.

**Un hash, c'est une chaîne de caractères** générée par une "fonction de hashage" à partir d'une chaîne de caractères en entrée. Ce hash est:

- **irréversible**: si `tartempion` est hashé en `xff3eoa42`, il est impossible de retrouver `tartempion` à partir de `xff3eoa42`.
- **unique** à une valeur d'entrée: le hash `xff3eoa42` ne peut être obtenu qu'avec l'entrée `tartempion`.

L'intérêt, c'est d'éviter de stocker des données sensibles: si il y a une fuite de données, les mots de passe de vos utilisateurs ne seront pas compromis. Par exemple, votre ordinateur ne sait pas quel est votre mot de passe: quand vous vous connectez, il calcule le hash de votre mot de passe et vérifie si cela correspond au hash qu'il a enregistré. Pour les mots de passe d'une application, c'est pareil !

(Mais par contre, il arrive qu'un algorithme de hashage soit "cracké": on peut trouver une manière d'obtenir le mot de passe à partir de son hash. Dans ce cas, il s'agit d'une grosse faille de sécurité.)

### Hasher un mot de passe

Écrire un algo de hashage dépasse très largement mes compétences. Fort heureusement, **Werkzeug (librairie sous-couche de Flask) a deux fonctions qui gèrent le hashage** pour nous:

- `generate_password_hash()`: créer un hash à partir d'un mot de passe (utilisé pour définir un mot de passe)
- `check_password_hash()`: vérifier que le mot de passe fourni est correct (utilisé quand on se connecte)


In [9]:
from werkzeug.security import generate_password_hash, check_password_hash

mdp = "tartempion"

mdp_hash = generate_password_hash(mdp)
print("le hash est:", mdp_hash)

print("check_password_hash avec le bon mdp:", check_password_hash(mdp_hash, mdp))
print("check_password_hash avec le mauvais mdp:", check_password_hash(mdp_hash, "ceci est une vilaine tentative d'intrusion"))

le hash est: scrypt:32768:8:1$IgwoEmxTo9osZ8b5$d1d064547264d128e68d055dd12ad07226b891692875ddc9568f4212e351b8f251e77caf438d1d4e8469db3d54c38b9b4f147b431ac434661702d4ba8a423eb5
check_password_hash avec le bon mdp: True
check_password_hash avec le mauvais mdp: False



---

# Créer un compte utilisateur depuis l'application

On a vu au dernier cours comment faire des *create* depuis l'application:
- on crée un modèle de base de données `User`
- on lui ajoute une méthode `User.create()` qui permette de créer un nouveau compte utilisateur
- on crée un formulaire WTForms nommé `UserCreateForm` pour créer un nouveau `User` depuis l'application
- on crée une template Jinja `user_create.html`
- on crée une route `user_create` qui permette de créer un `User`.

Dans l'application [`create_user`](./apps/s6/create_user/), on va donc créer ou modifier les fichiers suivants:

```txt
create/
└── app
    ├── app.py        # il faudra importer les routes de routes/users.py
    ├── models
    │   ├── forms.py  # on ajoute `UserCreateForm`
    │   └── users.py  # nouveau fichier qui stocke le modèle SQLAlchemy pour les `Users`
    ├── routes
    │   └── users.py  # nouveau fichier stockant toutes les routes relatives à la gestion d'utilisateurices
    └── templates
        └── pages
            └── user_create.html  # template HTML avec un formulaire pour créer des utilisateurices
```

## Créer un utilisateur depuis `User`:  `create`

Pour créer un `User` et le sauvegarder en base dans notre application, on va:
- **créer un fichier [`app/models/users.py`](./apps/s6/create_user/app/models/users.py)** qui contient notre modèle `User`
- **créer une fonction `User.create`** qui gère la création d'utilisateurs. `create` prend en paramètres `user_name`, `user_mail` et `user_password` et créé l'`user` si il n'existe pas déjà.

Regardons bien le code ci-dessous:

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]

    @staticmethod
    def create(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", str]]:
        """
        créer un nouveau user.

        notre fonction retourne:
        - (True, User) en cas de succès 
        - (False, <message d'erreur>) en cas d'erreur
        donc, le 1er item permet de savoir si l'insertion a fonctionné 
        """

        # on vérifie si il existe un autre `user` avec le même mail
        existing_user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().all()
        # l'user existe => on n'insère pas et on retourne un message d'erreur
        if len(existing_user):
            return False, f"Un utilisateur existe déjà pour le mail: {user_mail}"

        # si l'user n'existe pas, on le crée. remarquez l'utilisation de `generate_password_hash`
        new_user = User(
            user_name=user_name,
            user_mail=user_mail,
            user_password=generate_password_hash(user_password)
        ) 

        # pour finir, on insère l'user
        try:
            db.session.add(new_user)
            db.session.commit()
            return True, new_user
        except Exception as e:
            # en cas d'erreur au moment de l'insert, on retourne False et le message d'erreur de l'appli
            print(e)
            return False, "Erreur à la création du compte utilisateur"
```

Remarques:
- `User.create()` fonctionne **globalement globalement comme `Iconography.create`**: on reçoit les valeurs à sauvegarder en argument de la fonction, on crée l'objet (ici `User`), on l'insère dans un `try...except`
- **principale différence**: on ajoute une requête SQLAlchemy pour vérifier si un `user` n'existe pas avec le même mail

## Le formulaire `UserCreateForm`

On l'a dit, [`app/models/forms.py`](`./apps/s6/create_user/app/models/forms.py`) stockera le nouveau formulaire `UserCreateForm`.

Pour rappel, pour créer un nouveau `user` on a besoin d'un `user_mail`, d'un `user_name` et d'un `user_password`. Donc, **notre formulaire aura 3 champs obligatoires**.

**Voici `UserCreateForm`**, qui permet de créer un formulaire.

```py
from flask_wtf import FlaskForm
from wtforms import StringField, PasswordField
from wtforms.validators import DataRequired, Email, Length

class UserCreateForm(FlaskForm):
    user_name = StringField("Nom d'utilisateur.ice", validators=[DataRequired(), Length(max=50)])
    user_mail = StringField("Email", validators=[DataRequired(), Email()])
    user_password = PasswordField("Password", validators=[DataRequired(), Length(min=5, max=50)])
```

## La template `user_create.html`

La template HTML [`app/templates/pages/user_create.html`](./apps/s6/create_user/app/templates/pages/user_create.html) permet de **créer une interface utilisateur pour le formulaire**, via une template HTML. Voici son contenu:

```html
{% extends "base.html" %}

{% block title_extra %}| Créer un compte utilisateur{% endblock %}

{% block main_content %}
    <h1 class="title">Créer un compte utilisateur</h1>

    <form method="POST" action="{{ url_for('user_create') }}">
        {% with form=form, submit_label="Créer un compte" %}
            {% include "includes/form_fields.html" %}
        {% endwith %}
    </form>
{% endblock %}
```

**Remarques**: le formulaire utilise `form_fields.html`, et donc la template est quasiment identique à [app/templates/pages/icono_create.html](./apps/s6/create_user/app/templates/pages/icono_create.html).

## La route `user_create`

On a maintenant une classe formulaire et une template pour ce formulaire.

Maintenant, on va **créer la route `user_create` pour accéder au formulaire et pouvoir le soumettre**. Cette route se trouve dans [`app/routes/users.py`](./apps/s5/insert/app/routes/users.py)

```py
@app.route("/user/nouveau/", methods=["GET", "POST"])
def user_create():
    form = UserCreateForm()

    # form.validate_on_submit est True si:
    # - la requête est POST (on a soumis un formulaire)
    # - le formulaire est valide (wtforms a bien validé toutes les données fournies)
    if form.validate_on_submit():
        # on récupère les données et on les passe à create
        # `.data` permet de sélectionner la valeur fournie par l'utilisateur.ice
        user_name = form.user_name.data
        user_mail = form.user_mail.data
        user_password = form.user_password.data
        # `create` retourne:
        # - un booleen qui indique si la création réussi
        # - soit l'objet User crée, soit une liste d'erreurs
        success, data = User.create(
            user_name=user_name, 
            user_mail=user_mail, 
            user_password=user_password
        )
        # l'insert a réussi => rediriger sur la page d'accueil
        if success:
            flash("Compte utilisateur créé avec succès ! Vous pouvez maintenant vous connecter.", "success")
            return redirect("/")
        # l'insert a échoué => afficher les messages d'erreur.
        else: 
            data = "Les erreurs suivantes ont été repérées:" + ", ".join(data)
            flash(data, "error")
            return  render_template("pages/user_create.html", form=form, app_name=APP_NAME)
    return render_template("pages/user_create.html", form=form, app_name=APP_NAME)
```

**Explication de code**: comme pour toutes les routes de *create/update/delete*,
- **notre route accepte 2 méthodes HTTP**: cela est défini dans le `@app.route()` avec `methods=["GET", "POST"]`
    - `GET` est utilisé pour **accéder au formulaire** sans soumettre de données
    - `POST` est utilisé pour **soumettre le formulaire** (rappelez vous de la template, où on voit `<form method="POST">`)
- **la route a 2 branches correspondantes**
    - `if form.validate_on_submit()` confirme que on a envoyé une requête `POST` et que le formulaire est valide **=> on tente une insertion** avec `User.create()`
    - sinon, (c'est une requête `GET` pour accéder au formulaire ou les données ne sont pas valides), **on renvoie le formulaire** `user_create.html`.
- **`flash` est utilisé pour afficher les messages de succès ou d'erreur**. flash est une fonction Flask qui prend en 1er argument les messages à afficher, en 2e argument le statut du message (`success` ou `error`).

Et enfin, **on importe les routes de [`app/routes/users.py`](./apps/s6/create_user/app/routes/users.py)** dans [`app/app.py`](./apps/s6/create_user/app/app.py) en modifiant la dernière ligne du fichier:

```py
# l'ancien import était: `from app.routes import generic`
from app.routes import generic, users
```

## Tester le résultat

> **Lancer l'application `apps/s5/insert`**:
> ```py
> python ./apps/s5/insert/main.py
> ```

**Essayons de**:
- créer un nouveau `user`
- fournir des mauvaises données au formulaire pour voir comment elles sont gérées (mauvais format d'email, qui correspond déjà à un `user`...)



---

# Gestion d'utilisateurs

On peut maintenant créer des nouveaux `users`. Super ! Mais la gestion d'utilisateurs, ce n'est pas que pouvoir créer des comptes. C'est aussi:
- pouvoir **se connecter**
- **rester connecté.e** d'une page à l'autre
- **restreindre l'accès à certaines pages** seulement si on est connecté.e (dans notre cas: modifier la BDD seulement quand on est connecté.e).

## La gestion d'utilisateurs: Flask-Login

Flask-Login est un plugin qui gère la connexion d'utilisateurs. Pour l'utiliser, on doit:
- configurer un `LoginManager`
- augmenter `User` pour le rendre compatible avec Flask-Login
- définir un `user_loader` pour que l'appli Flask puisse déterminer les utilisateur.ice.s actuellement connecté.e.s

**Fichiers modifiés**:

```txt
apps/s6/login/
└── app
    ├── app.py        # on configure Flask-Login dans app.py
    └── models
        └── users.py  # on modifie la classe `User` pour que Flask-Login puisse interagir avec et on ajoute un `user_loader`
```

### Configurer le `LoginManager`

La première étape, c'est de compléter [`app/app.py`](./apps/s5/login/app/app.py):

```py
from flask_login import LoginManager

app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
# j'omets le reste de la config...
login_manager = LoginManager()
login_manager.init_app(app)

from app.routes import generic, users
```

`login_manager` est notre gestionnaire de connexions et il est totalement intégré à notre appli Flask grâce à `login_manager.init_app(app)`.

### Configurer l'`User`: `UserMixin`

Flask-Login a besoin que le modèle pour nos `Users` ait quelques propriétés bien définies pour fonctionner: `is_authenticated`, `is_active`, `is_anonymous`, `get_id` (qui permet de récupérer l'ID de l'utilisateur).

**Pas besoin de les définir à la main**: Flask-Login offre un `UserMixin` qui rajoute ces méthodes à notre modèle `User`. Pour ajouter le mixin, on modifie [`app/models/users.py`](./apps/s5/login/app/models/users.py):

```py
from flask_login import UserMixin

class User(db.Model, UserMixin):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
```

Et c'est tout ! Pour rappel, `User(db.Model, UserMixin)` signifie que **`User` hérite à la fois de `db.Model` et de `UserMixin`**. `UserMixin` définie les propriétés `is_authenticated`, `is_active`, `is_anonymous`, `get_id`m donc celles-ci deviennent accessibles depuis notre `User` (par exemple: `user.get_id()`).

### Définir un `user_loader`

Notre `LoginManager` défini dans `app/app.py` a besoin de pouvoir accéder à l'utilisateur actuellement connecté. **On définit donc une fonction qui permet d'accéder à un `User` par son ID**, toujours dans [`app/models/users.py`](./apps/s5/login/app/models/users.py):

```py
from app.app import login_manager

# ce décorateur signifie que la fonction `load_user` sera utilisée par le LoginManager pour accéder à l'user connecté.e
@login_manager.user_loader
def load_user(id_user: str):
    id_user = int(id_user)
    return db.session.get(User, id_user)
```

### Ce qui devient possible

Et voilà, Flask-Login est configuré.

Flask-Login offre des choses bien utile pour nous:
- une variable `current_user`, qui stocke l'`user` actuellement connecté.e dans une session (si il y a un `user` connecté)
- deux fonctions `login_user()` et `logout_user()` qui permettent de connecter/déconnecter un.e `user`.
- un décorateur `@login_required` qui permet de limiter l'accès à une route Flask aux utilisateur.ice.s connecté.es. 

## Login: se connecter

La logique à implémenter: 
- **une route** permet d'accéder à un formulaire pour se connecter
- pour se connecter à un compte existant, **l'utilisateur.ice fournit un mail et un mot de passe**
- **notre application vérifie** si ils correspondent à un `User` dans la base de données
- si oui, **on connecte l'utilisateur**

**Fichiers modifiés**:

```txt
apps/s6/login/
└── app
    ├── models
    │   ├── forms.py  # ajout du formulaire `UserLoginForm`
    │   └── users.py  # ajout d'une méthode pour identifier un `user` en fonction de son mail et mot de passe
    ├── routes
    │   └── users.py  # ajout d'une route pour le login
    └── templates
        └── pages
            └── user_login.html  # template avec un formulaire pour le login
```

### Formulaire Flask

Ensemble, ajoutons **un formulaire Flask `UserLoginForm`** pour se connecter. Pour rappel, on se connecte en utilisant son mail et son mdp (on peut utiliser `UserCreateForm` défini plus haut pour référence).


In [ ]:
# le code ici

(La solution est dans `apps/s5/login/app/models/forms.py`)

### Template `user_login.html`

[`app/templates/pages/user_login.html`](./apps/s5/login/app/templates/pages/user_login.html) reprend la même structure que toutes nos pages-formulaires:

```html
{% block main_content %}
    <h1 class="title">Se connecter</h1>

    <!-- comment interprétez vous le contenu de `form` ? -->
    <form method="POST" action="{{ url_for('user_login') }}">
        {% with form=form, submit_label="Connexion" %} 
            {% include "includes/form_fields.html" %}
        {% endwith %}
    </form>
{% endblock %}
```

Il faut maintenant **ajouter la logique côté base de données et une route pour se connecter**.

### Identifier un `User`

Dans [`app/models/users.py](./apps/s6/login/app/models/users.py), **on augmente `User` en ajoutant la méthode `get_user_by_credentials`**. Elle vérifier si un `user` existe pour le mail et le MDP fournis. Voilà le processus:
- l'utilisateur.ice fournit son mail et son mot de passe
- on hashe le mot de passe
- on vérifie qu'il y a une ligne dans la base de données avec le bon mail et le bon hash de mot de passe.
- si oui, on retourne le `user`, sinon on retourne `None`.

```py
class User(db.Model, UserMixin):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    
    @staticmethod
    def create(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", List[str]]]:
        ...
    
    @staticmethod
    def get_user_by_credentials(user_mail: str, user_password: str) -> Optional["User"]:
        """
        identifier un User par son mail et son mdp. 

        :returns: l'User si le mail et mdp sont valides, None sinon 
        """
        # retourne soit un `User`, soit None 
        user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().first()
        # on vérifie que le `User` avec ce mail a bien le bon mot de passe
        if user and check_password_hash(user.user_password, user_password):
            return user
        # sinon, on retourne None
        return None
```

### Une route pour se connecter

Pour finir, on ajoute une route `/user/connexion` pour se connecter.

Voici la route dans [`app/routes/users.py`](./apps/s5/login/app/routes/users.py):

```py
from flask_login import login_user

# comment interpréter ce qui se passe dans cette route ?
@app.route("/user/connexion/", methods=["GET", "POST"])
def user_login():
    form = UserLoginForm()

    if current_user.is_authenticated:
        flash("Vous êtes déjà connecté.e", "success")
        return redirect("/")

    if form.validate_on_submit():
        user_mail = form.user_mail.data
        user_password = form.user_password.data
        user = User.get_user_by_credentials(
            user_mail=user_mail,
            user_password=user_password
        )
        if user:
            flash("Vous êtes maintenant connecté.e", "success")
            login_user(user)
            return redirect("/")
        else:
            flash("Identifiants incorrects", "error")
            return render_template("pages/user_login.html", form=form, app_name=APP_NAME)
        
    return render_template("pages/user_login.html", form=form, app_name=APP_NAME)
```

## Logout: se déconnecter

De la même manière que l'on a créé une route pour se connecter, on en crée une pour se déconnecter. C'est bien plus simple: on veut une simple requête `GET` à `user/deconnexion/`, et cette requête exécute la fonction Flask-Login `logout_user`:

On ajoute donc à  [`app/routes/users.py`](./apps/s5/login/app/routes/users.py) la fonction suivante. On remarque l'usage du décorateur `@login_required`: il n'est possible d'accéder à cette route que si on est connecté.e.

```py
@app.route("/user/deconnexion/")
@login_required
def user_logout():
    logout_user()
    flash("Vous êtes déconnecté.e")
    return redirect("/")
```

## Ajouter `@login_required` aux routes *create/update/delete*

Pour empêcher que des personnes non-connectées ne fassent n'importe quoi, on ajoute le décorateur `@login_required` à toutes les routes *create/update/delete* (sauf `user_create`, qui permet de créer son compte utilisateur).

On modifie donc les routes suivantes dans [`app/routes/generic.py`](./apps/s6/login/app/routes/generic.py) 

```py
@app.route("/iconographie/nouveau", methods=["GET", "POST"])
# on ajoute `login_required`: on ne peut créer une ressource icono que si on est connecté.e
@login_required
def icono_create():
    # ... le reste de la fonction ne change pas, je l'omets donc ici

@app.route("/iconographie/<int:id_icono>/modifier", methods=["GET", "POST"])
@login_required
def icono_update(id_icono: int):
    # ...

@app.route("/iconography/<int:id_icono>/supprimer", methods=["GET", "POST"])
@login_required
def icono_delete(id_icono: int):
    # ...
```

## Voir le résultat

Allons maintenant voir le résultat.

> **On lance l'application [`login`](./apps/s6/login/)**:
> ```py
> python apps/s6/login/main.py
> ```

Essayons de:
- créer un compte
- se connecter
- mettre les mauvais *credentials* (mail + mot de passe)
- accéder aux routes *create/update/delete* sans être connecté.e
- ...